In [1]:
import os
import sys
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# =========================================================================
# 0. DIRECTORY PATH FIX 
# =========================================================================
# This appends the parent folder (the root of the repo) to Python's path
# so it can successfully find and import the 'models' directory from inside 
# your 'research' folder.
sys.path.append(os.path.abspath('..')) 

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder, UNET_2D_simp
from models.frameworks import IsoAlign

In [2]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Import the core network structures from the repository's modules
from models.models_nc import ResNet1D, FourierEncoder
from models.frameworks import IsoAlign

# =========================================================================
# NEW: SEQUENTIAL FUSION BiLSTM FOR WAVELETS (Foolproof Shape Router)
# =========================================================================
class SequentialFusionBiLSTM(nn.Module):
    def __init__(self, in_channels, scales, hidden_dim, out_dim, time_steps=48):
        super().__init__()
        self.in_channels = in_channels
        self.scales = scales
        self.time_steps = time_steps # Explicitly track the 48 time steps
        
        # Flattened dimension per time step: C x Scales (3 * 64 = 192)
        self.input_dim = in_channels * scales 
        
        # Sequence Modeling: BiLSTM to capture forward/backward dynamics
        self.bilstm = nn.LSTM(
            input_size=self.input_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )
        
        # Project the BiLSTM output to match the IsoAlign latent dimension
        self.fc = nn.Linear(hidden_dim * 2, out_dim)

    def forward(self, x):
        # The parent IsoAlign framework is shuffling the dimensions upstream.
        # We will dynamically hunt down where it put our C(3), Scales(64), and L(48)
        # using their unique sizes.
        dims = list(x.shape)
        B = dims[0]
        
        # Find the axes by looking for their size (starting from index 1 to skip Batch)
        c_idx = dims.index(self.in_channels, 1)
        s_idx = dims.index(self.scales, 1)
        l_idx = dims.index(self.time_steps, 1)
        
        # Force the tensor into EXACTLY: (Batch, Time, Channels, Scales)
        # This guarantees it becomes (B, 48, 3, 64) regardless of upstream transposes
        x = x.permute(0, l_idx, c_idx, s_idx).contiguous() 
        
        # Flatten the spatial/channel dimensions safely
        # New shape: (B, 48, 192)
        x = x.view(B, self.time_steps, self.input_dim) 
        
        # Sequence Modeling
        out, _ = self.bilstm(x) # out shape: (B, L, hidden_dim * 2)
        
        # Pool across the temporal dimension (L) to get a single vector per batch item
        out_pooled = torch.max(out, dim=1)[0]
        
        # Map to LATENT_DIM
        final_rep = self.fc(out_pooled)
        
        return None, final_rep
# =========================================================================
# 1. FIXED DATASET CLASS
# =========================================================================
class SleepEDF_HF_Dataset(Dataset):
    def __init__(self, pt_file_path="/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt", split="train"):
        print(f"Loading Hugging Face dataset from {pt_file_path}...")
        
        data_obj = torch.load(pt_file_path, map_location="cpu")
        
        if isinstance(data_obj, dict):
            if split in data_obj and isinstance(data_obj[split], dict):
                data_obj = data_obj[split]
                
            if "samples" in data_obj:
                self.data = data_obj["samples"]
            elif "data" in data_obj:
                self.data = data_obj["data"]
            elif "x_data" in data_obj:
                self.data = data_obj["x_data"]
            elif "X_train" in data_obj:
                self.data = data_obj["X_train"]
            else:
                raise ValueError(f"Could not find the tensor. Available keys: {data_obj.keys()}")
        else:
            self.data = data_obj
            
        if not isinstance(self.data, torch.Tensor):
            self.data = torch.FloatTensor(self.data)
        else:
            self.data = self.data.float()
            
        if self.data.dim() == 2:
            self.data = self.data.unsqueeze(1)
            
        print(f"✅ Successfully loaded {len(self.data)} epochs from split '{split}'.")
        self.window = torch.hann_window(128)
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        x_t = self.data[idx] 
        x_time = x_t 
        
        x_fft = torch.fft.rfft(x_t, dim=-1)
        magnitude = torch.abs(x_fft)
        phase = torch.angle(x_fft)
        x_fourier = torch.cat([magnitude, phase], dim=0)
        
        x_stft = torch.stft(x_t, n_fft=128, hop_length=64, window=self.window, return_complex=True)
        x_wavelet = torch.abs(x_stft)[:, :64, :] 
        x_wavelet = F.pad(x_wavelet, (0, 1)) 
        
        return x_time, x_fourier, x_wavelet

# =========================================================================
# 2. SETUP HYPERPARAMETERS & DEVICE CONFIGURATION
# =========================================================================
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu") 
print(f"\nUsing device: {DEVICE}")

N_CHANNELS = 3 
TIME_CHANNELS = N_CHANNELS
FT_CHANNELS = N_CHANNELS * 2 

BATCH_SIZE = 64
EPOCHS = 40
TIME_STEPS = 3000    
LATENT_DIM = 128     

class Args:
    wo_OB = False
    wo_OF = False
args = Args()

# =========================================================================
# 3. INITIALIZE DATA LOADERS
# =========================================================================
dataset_path = "/home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt"
train_dataset = SleepEDF_HF_Dataset(pt_file_path=dataset_path, split="train")
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)

_, _, sample_w = train_dataset[0]
SPECT_FREQ = sample_w.shape[1]   
SPECT_TIME = sample_w.shape[2]   

print(f"Spectrogram dimensions configured to: {SPECT_FREQ} Freqs x {SPECT_TIME} Time steps")

# =========================================================================
# 4. INSTANTIATE ARCHITECTURE COMPONENTS
# =========================================================================
# A. Time Encoder (1D ResNet)
time_encoder = ResNet1D(
    in_channels=TIME_CHANNELS, base_filters=32, kernel_size=5, stride=1, groups=1, 
    n_block=3, n_classes=LATENT_DIM, downsample_gap=2, increasefilter_gap=4, 
    use_do=True, backbone=True, output_dim=LATENT_DIM
).to(DEVICE)

# B. Wavelet Encoder (NEW: Sequential Fusion BiLSTM)
# Replaces the previous UNET_2D_simp implementation
spect_encoder = SequentialFusionBiLSTM(
    in_channels=N_CHANNELS, 
    scales=SPECT_FREQ, 
    hidden_dim=64, 
    out_dim=LATENT_DIM
).to(DEVICE)

# C. Fourier Encoder Wrapper
class FourierWrapper(nn.Module):
    def __init__(self, in_length):
        super().__init__()
        self.enc = FourierEncoder(in_channels=FT_CHANNELS, in_length=in_length, out_channels=LATENT_DIM)
        if in_length == 3000:
            self.enc.fc_abs = nn.Linear(376, 1)
            self.enc.fc_angle = nn.Linear(376, 1)
            
    def forward(self, x):
        return None, self.enc(x)

ft_encoder = FourierWrapper(in_length=TIME_STEPS).to(DEVICE)

# D. Master IsoAlign Multi-View Framework
model = IsoAlign(
    backbone=time_encoder, spect_encoder=spect_encoder, FT_encoder=ft_encoder, 
    DEVICE=DEVICE, dim=LATENT_DIM, batch_size=BATCH_SIZE, args=args
).to(DEVICE)

optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)




Using device: cuda:0
Loading Hugging Face dataset from /home/gella.saikrishna/code/Learning-with-FrameProjections/Research/data/sleep_multichannel_3c.pt...
✅ Successfully loaded 265575 epochs from split 'train'.
Spectrogram dimensions configured to: 64 Freqs x 48 Time steps


In [ ]:
print("\n🚀 Starting Self-Supervised Sequential Fusion Pre-training on GPU-1...")

checkpoint_dir = "checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)

best_loss = float('inf')
save_every_n_epochs = 5 
log_interval = 50

start_epoch = 0
resume_path = os.path.join(checkpoint_dir, "best_sequential_bilstm.pth") 

if os.path.exists(resume_path):
    print(f"\n🔄 Found checkpoint at {resume_path}. Loading...")
    checkpoint = torch.load(resume_path, map_location=DEVICE)
    
    # Extract the state dict (handles both 'best' weights and full recovery checkpoints)
    state_dict = checkpoint.get('model_state_dict', checkpoint)
    
    # Strip the _orig_mod. prefix
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith("_orig_mod."):
            name = k[len("_orig_mod."):]
        else:
            name = k
        new_state_dict[name] = v
        
    model.load_state_dict(new_state_dict) 
    print("✅ Model weights loaded successfully.")
else:
    print("\n⚠️ No checkpoint found. Starting training from scratch.")

if hasattr(model, 'bilstm'):
    model.bilstm.flatten_parameters()

model = torch.compile(model)
scaler = torch.amp.GradScaler("cuda")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0
    
    for batch_idx, (batch_t, batch_f, batch_w) in enumerate(train_loader):
        batch_t = batch_t.to(DEVICE, non_blocking=True)
        batch_f = batch_f.to(DEVICE, non_blocking=True)
        batch_w = batch_w.to(DEVICE, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.amp.autocast("cuda"):
            loss = model(batch_t, batch_w, batch_f)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item()
        
        if (batch_idx + 1) % log_interval == 0:
            current_batch_loss = loss.item()
            running_avg_loss = total_loss / (batch_idx + 1)
            
            print(f"  Epoch: {epoch+1} [{batch_idx + 1}/{len(train_loader)}] | Batch Loss: {current_batch_loss:.4f} | Running Avg: {running_avg_loss:.4f}")
            
    avg_loss = total_loss / len(train_loader)
    print(f"=== Epoch [{epoch+1}/{EPOCHS}] Completed | Final Average Loss: {avg_loss:.4f} ===\n")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
        best_model_path = os.path.join(checkpoint_dir, "best_sequential_bilstm.pth")
        torch.save(model.state_dict(), best_model_path)
        print(f"   🌟 New best loss achieved! Saved encoder to: {best_model_path}")
        
    if (epoch + 1) % save_every_n_epochs == 0:
        checkpoint_path = os.path.join(checkpoint_dir, f"sequential_bilstm_epoch_{epoch+1}.pth")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss,
        }, checkpoint_path)
        print(f"   (i) Full recovery checkpoint saved to: {checkpoint_path}")

print("\n🎉 Pre-training Complete!")


🚀 Starting Self-Supervised Sequential Fusion Pre-training on GPU-1...

🔄 Found checkpoint at checkpoints/best_sequential_bilstm.pth. Loading...
✅ Model weights loaded successfully.


W0707 03:54:52.854000 348579 torch/_inductor/utils.py:1137] [0/0_1] Not enough SMs to use max_autotune_gemm mode
/home/gella.saikrishna/.venv/lib/python3.12/site-packages/torch/_inductor/lowering.py:1625: FutureWarning: `torch._prims_common.check` is deprecated and will be removed in the future. Please use `torch._check*` functions instead.
  check(
/home/gella.saikrishna/.venv/lib/python3.12/site-packages/torch/_inductor/lowering.py:1625: FutureWarning: `torch._prims_common.check` is deprecated and will be removed in the future. Please use `torch._check*` functions instead.
  check(


  Epoch: 1 [50/4149] | Batch Loss: 3.2738 | Running Avg: 4.1038
  Epoch: 1 [100/4149] | Batch Loss: 2.5864 | Running Avg: 3.4540
  Epoch: 1 [150/4149] | Batch Loss: 2.9582 | Running Avg: 3.2239
  Epoch: 1 [200/4149] | Batch Loss: 2.8173 | Running Avg: 3.1235
  Epoch: 1 [250/4149] | Batch Loss: 2.7735 | Running Avg: 3.0651
  Epoch: 1 [300/4149] | Batch Loss: 3.0006 | Running Avg: 3.0312
  Epoch: 1 [350/4149] | Batch Loss: 2.2976 | Running Avg: 2.9833
  Epoch: 1 [400/4149] | Batch Loss: 2.8632 | Running Avg: 2.9534
  Epoch: 1 [450/4149] | Batch Loss: 2.4811 | Running Avg: 2.9275
  Epoch: 1 [500/4149] | Batch Loss: 2.8559 | Running Avg: 2.9117
  Epoch: 1 [550/4149] | Batch Loss: 3.3284 | Running Avg: 2.9349
  Epoch: 1 [600/4149] | Batch Loss: 2.8195 | Running Avg: 2.9389
  Epoch: 1 [650/4149] | Batch Loss: 2.5846 | Running Avg: 2.9222
  Epoch: 1 [700/4149] | Batch Loss: 4.0487 | Running Avg: 2.9240
  Epoch: 1 [750/4149] | Batch Loss: 2.8875 | Running Avg: 2.9274
  Epoch: 1 [800/4149] | Ba